# EQI-Only Feature Model — Methodological Sensitivity Analysis

**Purpose:** Test how well environmental features alone predict chronic disease prevalence, removing all features that overlap with the PLACES model's own predictors.

**Why this matters:** PLACES tract-level estimates are generated using a multilevel regression model that incorporates age, sex, race/ethnicity, education, and county-level poverty. Including those same variables (or close proxies) as features in our model partially recovers relationships that were baked into the outcome itself, inflating R². This notebook isolates the genuine environmental signal.

**Removed features (overlap with PLACES predictors):**
- `poverty` — PLACES uses county-level poverty directly
- `bs_25yo` — overlaps with PLACES education predictor
- `unemployment` — strongly correlated with poverty
- `rent_income_pct` — correlated with poverty
- All `PLACES_*` features (other diseases used as features in main model)

**Retained features (EQI environmental + built environment):**
- Air: `ozone`, `pm25`, `nata_sum`
- Land/water: `pest`
- Built environment: `facilities_area`, `education_area`, `food_ratio`, `nindex_open`, `selfservice`
- Sociodemographic context (not in PLACES model): `X_pop_dens`, `time2work`, `crime_index`, `RUCA_pri_cat`

**Targets:** STROKE, KIDNEY, CHD, CANCER (same as main model)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

cdc_epa_data = pd.read_csv(
    "/Users/virginiaberthy/UVA-MSDS/DS 5003 - Healthcare DS - SP '26/Group Project/FINAL MODEL & RUN/EQI_2011_2015_PLACES_2020_merged_copy.csv"
)
print(f"Loaded: {cdc_epa_data.shape[0]:,} rows, {cdc_epa_data.shape[1]} columns")

In [ ]:
# EQI-only feature set — no overlap with PLACES predictors, no PLACES features
eqi_features = [
    # Air quality
    'ozone', 'pm25', 'nata_sum',
    # Land
    'pest',
    # Built environment
    'facilities_area', 'education_area', 'food_ratio', 'nindex_open', 'selfservice',
    # Sociodemographic context (not used by PLACES)
    'X_pop_dens', 'time2work', 'crime_index', 'RUCA_pri_cat',
]

targets = ['PLACES_STROKE', 'PLACES_KIDNEY', 'PLACES_CHD', 'PLACES_CANCER']
id_cols = ['tract_fips', 'State', 'County', 'StateAbbr', 'CountyName']

cols_to_keep = eqi_features + targets + id_cols
cdc_epa_selected = cdc_epa_data[cols_to_keep]

# stratified 80/20 split by state — same random state as main model for comparability
cdc_epa_train, cdc_epa_test = train_test_split(
    cdc_epa_selected, test_size=0.2, random_state=5003,
    stratify=cdc_epa_selected['State']
)

feature_cols = eqi_features
num_cols = [c for c in feature_cols if cdc_epa_train[c].dtype in ['float64', 'int64']]

print(f"EQI-only features: {len(feature_cols)}")
print(f"Targets: {len(targets)}")
print(f"Train: {len(cdc_epa_train):,}  Test: {len(cdc_epa_test):,}")

In [ ]:
# same hyperparameter grid as main v2 model for fair comparison
xgb_params_v2 = {
    'n_estimators': [400, 500, 600, 800],
    'max_depth': [6, 7, 8, 10],
    'learning_rate': [0.03, 0.05, 0.07],
    'subsample': [0.75, 0.8, 0.85],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0.5, 1, 2, 5],
    'gamma': [0, 0.1, 0.3],
}

results_eqi = []

for target in targets:
    X_train = cdc_epa_train[feature_cols].copy()
    y_train = cdc_epa_train[target]
    X_test = cdc_epa_test[feature_cols].copy()
    y_test = cdc_epa_test[target]

    imputer = SimpleImputer(strategy='median')
    X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
    X_test[num_cols] = imputer.transform(X_test[num_cols])

    scaler = MinMaxScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])

    xgb_cv = RandomizedSearchCV(
        XGBRegressor(random_state=5003),
        xgb_params_v2, n_iter=100, cv=5, scoring='r2',
        random_state=5003, n_jobs=-1
    )
    xgb_cv.fit(X_train, y_train)

    y_pred = xgb_cv.predict(X_test)
    rmse = round(np.sqrt(mean_squared_error(y_test, y_pred)), 4)
    r2 = round(r2_score(y_test, y_pred), 4)

    print(f"{target.replace('PLACES_', ''):12s}  R²: {r2}  RMSE: {rmse}")

    results_eqi.append({
        'Target': target.replace('PLACES_', ''),
        'R²_EQI_only': r2,
        'RMSE_EQI_only': rmse,
    })

results_eqi_df = pd.DataFrame(results_eqi).sort_values('R²_EQI_only', ascending=False)
results_eqi_df

## Comparison to Full Feature Model

Paste in the R² and RMSE values from your main v2 model (with all features) to see the performance gap. The drop tells you how much of the predictive power was coming from the demographic/PLACES-overlap features versus genuine environmental signal.

In [ ]:
# fill in your main model results here for comparison
main_results = pd.DataFrame({
    'Target': ['STROKE', 'KIDNEY', 'CHD', 'CANCER'],
    'R²_full':   [0.9903, 0.9868, 0.9865, 0.9839],   # update with your final v2 numbers
    'RMSE_full': [0.1233, 0.0953, 0.2586, 0.2354],   # update with your final v2 numbers
})

comparison = main_results.merge(results_eqi_df, on='Target')
comparison['R²_drop'] = (comparison['R²_full'] - comparison['R²_EQI_only']).round(4)
comparison['RMSE_increase'] = (comparison['RMSE_EQI_only'] - comparison['RMSE_full']).round(4)
comparison